# 02 - Regridding to a common 1 deg grid (global)   [PLAN A]

- EC-Earth3 historical and all GCM runs (2025 / 2050, hosing / no-hosing): 0.7 deg -> 1 deg
- ERA5: 0.25 deg -> 1 deg
- **All on the same grid** (same latitudes, same longitudes), so that step1 compares identical pixels.

**Method: conservative** (average weighted by overlap areas). This is the appropriate method for precipitation:
the amount of water is conserved (checked below). No bilinear interpolation (it smooths extremes and creates
phantom rain). Same method for every dataset.

RCMs are not regridded here (they stay at 0.25 deg).

No land/sea mask: regridding is done over the whole globe.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']

TARGET_RES = 1.0                       # degrees
ERA5_PERIOD = ("1981", "2025")         # ERA5: only regrid what is needed
DATASETS_TO_REGRID = ["era5", "gcm_historical", "gcm_2025_nohosing", "gcm_2025_hosing",
                      "gcm_2050_nohosing", "gcm_2050_hosing"]
OVERWRITE = False                      # False = skip files that already exist
REGRID_DIR = bu.out_dir(cfg, "plan_A", "regrid_1deg")
dst_lat, dst_lon = bu.target_grid(TARGET_RES)
print(f"Target grid: {len(dst_lat)} x {len(dst_lon)}  (lat {dst_lat[0]}..{dst_lat[-1]}, lon {dst_lon[0]}..{dst_lon[-1]})")

## 1. Regridding (one file per dataset and per variable)

In [ ]:
done, skipped = [], []
for name in DATASETS_TO_REGRID:
    for var in VARS:
        path = REGRID_DIR / f"{name}_{var}_1deg.nc"
        if path.exists() and not OVERWRITE:
            skipped.append(path.name); continue
        if not bu.dataset_available(cfg, name, var):
            print(f"[missing] {name}/{var} - fill in the path in notebook 00")
            continue
        print(f"-> {name}/{var}")
        da = bu.read_raw(cfg, name, var)
        if name == "era5":
            da = bu.subset_time(da, *ERA5_PERIOD)
        out = bu.regrid_conservative(da, dst_lat, dst_lon)
        out.attrs.update({"source_dataset": name, "target_res_deg": TARGET_RES})
        bu.save_da(out.rename(var), path, name=var)
        done.append(path.name)
print("\ndone:", len(done), "| already present:", len(skipped))

## 2. Checks

1. all outputs have **exactly** the same grid;
2. conservation of the spatial mean (what goes in = what comes out);
3. overview of the raw EC-Earth3 historical - ERA5 difference at 1 deg.

In [ ]:
files = sorted(REGRID_DIR.glob("*_1deg.nc"))
ref_grid = None
bad = []
for f in files:
    da = bu.read_processed(f)
    if ref_grid is None:
        ref_grid = da
    elif not bu.same_grid(da, ref_grid):
        bad.append(f.name)
print(f"{len(files)} files; different grid: {bad if bad else 'none'}")
assert not bad, "Grids are not all identical"

In [ ]:
rows = []
for name in DATASETS_TO_REGRID:
    for var in VARS:
        f = REGRID_DIR / f"{name}_{var}_1deg.nc"
        if not f.exists() or not bu.dataset_available(cfg, name, var):
            continue
        raw = bu.read_raw(cfg, name, var).isel(time=slice(0, 60))
        new = bu.read_processed(f, var).isel(time=slice(0, 60))
        a = float(bu.area_mean(raw).mean().compute())
        b = float(bu.area_mean(new).mean().compute())
        rows.append({"dataset": name, "var": var, "global mean native": round(a, 4),
                     "global mean 1deg": round(b, 4), "relative difference (%)": round(100 * (b - a) / (abs(a) + 1e-9), 3)})
pd.DataFrame(rows)

*(Global means: the difference should be close to 0 %, since conservative regridding preserves the amount of water. Small differences can come from cells near the poles or from missing values.)*

In [ ]:
# Overview of the raw bias at 1 deg: EC-Earth3 historical - ERA5, mean 1981-2014
P = ("1981", "2014")
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, var in zip(axes, VARS):
    fe, fg = REGRID_DIR / f"era5_{var}_1deg.nc", REGRID_DIR / f"gcm_historical_{var}_1deg.nc"
    if not (fe.exists() and fg.exists()):
        ax.set_title(f"{var}: missing files"); continue
    e = bu.subset_time(bu.read_processed(fe, var), *P).mean("time").compute()
    g = bu.subset_time(bu.read_processed(fg, var), *P).mean("time").compute()
    d = (g - e) if var != "pr" else 100 * (g - e) / (e + 1e-3)
    lim = float(np.nanpercentile(np.abs(d), 98))
    im = ax.pcolormesh(d.lon, d.lat, d, shading="auto", cmap="RdBu_r", vmin=-lim, vmax=lim)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f"{var}: GCM - ERA5 ({'%' if var == 'pr' else 'degC'})")
plt.tight_layout(); plt.savefig(bu.out_dir(cfg, "plan_A", "exploration") / "raw_bias_1deg.png", dpi=110); plt.show()